In [1]:
# import libraries
import pandas as pd
import geopandas as gpd
import ee
import geemap

import numpy as np

## Connect to Google Earth Engine (GEE)

In [2]:
# Authenticate GEE
ee.Authenticate()

# Initialize GEE
EE_PROJECT_ID = ""   # Change to your project ID

# ee.Initialize(project=EE_PROJECT_ID)
ee.Initialize()

## Map to Visualize Layers

In [3]:
# Center coordinates to show map
kano_center =  (11.999986,  8.551571)

In [4]:
# Creat map to visualize Kano boundary data
aoi_map = geemap.Map(center=kano_center, zoom=10)
aoi_map.add_basemap("SATELLITE")

# Create a map to visualise S2 image & boundary data
s2_map = geemap.Map(center=kano_center, zoom=6)
s2_map.add_basemap("SATELLITE")

# Create maps to visualise thematic layers
thematic_map_1 = geemap.Map(center = kano_center, zoom=8)
thematic_map_1.add_basemap("SATELLITE")

thematic_map_2 = geemap.Map(center = kano_center, zoom=8)
thematic_map_2.add_basemap("SATELLITE")

## Visualization Parameters

In [5]:
# Boundary visualization params 
vis_params_fao_1 = {
  "fillColor": 'b5ffb4',
  "color": '00909F',
  "width": 1.0,
}

vis_params_aoi = {"fillcolor": "", "color": "red"}

# Sentinel-2 Visualization parameters
vis_params_s2_rgb = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['B4', 'B3', 'B2'],
    "gamma" : 0.9
}

vis_params_s2_fc = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['B8', 'B4', 'B3'],
    "gamma" : 0.9
}


# Spectral indices visualization params
vis_params_ndvi = {"min": 0, "max":0.5, "palette":['red', 'orange', 'yellow',  'lightgreen', 'green' ]}

vis_params_ndwi = {
    "min": -0.5,
    "max": 0.5,
    "palette": ["white", "lightcyan", "deepskyblue", "dodgerblue", "blue"]}


# DEM & Slope visualization params
vis_params_dem = {'min': 0, 'max': 500, "palette": ["Red", "Green", "Yellow"]}
vis_params_slope = {'min': 0, 'max': 10, "palette": ["Red", "Green", "Yellow"]}



In [6]:
# Precipitation vis params
vis_params_precp = {"min": 300, "max": 1400, "palette": ["white", "lightblue", "deepskyblue", "blue", "darkblue"]}

# Land cover
vis_params_esa_lc = {'bands': ['Map']}


# Soil texture
vis_params_soil = {
    "min": 1,
    "max": 12,
    "palette": [
        "f7fcf5", "e5f5e0", "c7e9c0", "a1d99b",
        "74c476", "41ab5d", "238b45", "006d2c",
        "fdae6b", "fd8d3c", "e6550d", "a63603"
    ]
}


# Drainage Density
vis_params_drainage = {
    "min": 0,
    "max": 30,
    "palette": ["white", "lightblue", "deepskyblue", "blue", "darkblue"]
}

#TWI
vis_params_twi = {"min": 5, "max": 50, "palette": ["brown", "yellow", "lightgreen", "green", "darkgreen"]}


## Helper Functions

In [7]:
# Function to export Image to Drive
def export_image_to_drive(image: ee.Image, description: str, folder: str, extent: ee.Geometry, scale: int, crs: str = 'EPSG:4326'):
    """
    Starts a Google Earth Engine batch task to export a raster image to Google Drive.

    Args:
        image (ee.Image): The Earth Engine image to be exported.
        description (str): A human-readable name for the task and the output file prefix.
        folder (str): The name of the Google Drive folder where the file will be saved.
        extent (ee.Geometry): The regional boundary (AOI) to clip the export to.
        scale (int): The resolution in meters per pixel (e.g., 10 for Sentinel-2).
        crs (str): The coordinate reference system.

    Returns:
        None: The function initiates an asynchronous background task.
    """
    # Convert all bands to Float32 (or Float64)
    image = image.toFloat()

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=folder,
        fileNamePrefix=description,
        region=extent,
        scale=scale,
        crs=crs,
        maxPixels=1e13
    )
    task.start()
    return task
    print(f"Started export: {description}")

## Boundary Data

In [8]:
# Download admin boundaries & visualise
# Reference link for data: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_SIMPLIFIED_500m_2015_level0

fao_gaul_l0 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level0') # Country boundaries
fao_gaul_l1 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level1') # State boundaries
fao_gaul_l2 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level2') # LGA boundaries

aoi_map.addLayer(fao_gaul_l0, {}, 'Country Boundaries')
aoi_map.addLayer(fao_gaul_l1, {}, 'State Boundaries')
aoi_map.addLayer(fao_gaul_l2, {}, 'LGA Boundaries')
aoi_map

Map(center=[11.999986, 8.551571], controls=(WidgetControl(options=['position', 'transparent_bg'], position='to…

In [9]:
# Check columns
print(fao_gaul_l0.limit(0).getInfo()["columns"])

# Extract Nigerian boundaries from FAO GAUL
nga_l0 = fao_gaul_l0.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # Single Nigeria boundary
nga_l1 = fao_gaul_l1.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # State boundaries in Nigeria
nga_l2 = fao_gaul_l2.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # LGAs boundaries in Nigeria

kano_l0 = nga_l1.filter(ee.Filter.eq("ADM1_NAME", "Kano")) # Kano boundary/extent
aky_lga = nga_l2.filter(ee.Filter.eq("ADM2_NAME", "Akinyele")) # Akinyele LGA in Ibadan


# Get geometry from Kano boundary (FeatureCollection)
aoi = kano_l0.geometry()
aoi_bbox = aoi.bounds()

aoi_map.addLayer(kano_l0, vis_params_aoi, 'Kano Boundary')
aoi_map.addLayer(aky_lga, vis_params_aoi, 'Akinyele LGA')
aoi_map

{'ADM0_CODE': 'Integer', 'ADM0_NAME': 'String', 'DISP_AREA': 'String', 'EXP0_YEAR': 'Integer', 'STATUS': 'String', 'STR0_YEAR': 'Integer', 'Shape_Area': 'Float', 'Shape_Leng': 'Float', 'system:index': 'String'}


Map(center=[11.999986, 8.551571], controls=(WidgetControl(options=['position', 'transparent_bg'], position='to…

## Explore Image operations

In [10]:
# Explore Sentinel-2 image collection
s2_img_col = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') # All S2 images for the entire world
    .filterDate('2020-01-01', '2020-01-30') # Limite to specific period/date 
    .filterBounds(aoi) # Limit search to Kano
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) # Cloud percentage < 30% 
)


# Apply scaling factor
def apply_scale(image):
    scaled_image = image.multiply(0.0001).copyProperties(image, ['system:time_start'])
    return  scaled_image

s2_img_col = s2_img_col.map(apply_scale)


# Check number of images in 's2_img_col'
print(f"Number of images in S2 collection: {s2_img_col.size().getInfo()}\n")

# Check properties of the 's2_img_col'
print(s2_img_col.getInfo())

# Take only the first image in 's2_img_col'
first_s2_img = s2_img_col.first()
s2_img_bands = first_s2_img.bandNames().getInfo()

print(f"First image in S2 collection: {first_s2_img.getInfo()}\n")
print(f"Bands in S2 images: {s2_img_bands}\n")


# Select only RGB bands
first_s2_img_rgb = first_s2_img.select(["B2", "B4"])
print(f"Bands in S2 images [RGB]: {first_s2_img_rgb.bandNames().getInfo()}\n")


# Add layers to map
s2_map.addLayer(kano_l0, vis_params_aoi, "Kano Boundary")
s2_map.addLayer(first_s2_img.clip(kano_l0.geometry()), vis_params_s2_rgb, "S2 First Image")
s2_map

Number of images in S2 collection: 51

{'type': 'ImageCollection', 'bands': [], 'version': 1784978571952087, 'id': 'COPERNICUS/S2_SR_HARMONIZED', 'properties': {'date_range': [1490659200000, 1647907200000], 'period': 0, 'system:visualization_0_min': '0.0', 'type_name': 'ImageCollection', 'keywords': ['copernicus', 'esa', 'eu', 'msi', 'reflectance', 'sentinel', 'sr'], 'system:visualization_0_bands': 'B4,B3,B2', 'thumb': 'https://mw1.google.com/ges/dd/images/COPERNICUS_S2_SR_thumb.png', 'description': '<p>Sentinel-2 is a wide-swath, high-resolution, multi-spectral\nimaging mission supporting Copernicus Land Monitoring studies,\nincluding the monitoring of vegetation, soil and water cover,\nas well as observation of inland waterways and coastal areas.</p><p>The Sentinel-2 L2 data are downloaded from scihub. They were\ncomputed by running sen2cor. WARNING: ESA did not produce L2 data\nfor all L1 assets, and earlier L2 coverage is not global.</p><p>The assets contain\n12 UINT16 spectral ban

Map(center=[11.999986, 8.551571], controls=(WidgetControl(options=['position', 'transparent_bg'], position='to…

In [11]:
# Mosaic the entire S2 collection & clip to Kano extent 
s2_mosaic = s2_img_col.mosaic()
s2_mosaic_clipped = s2_mosaic.clip(aoi)
print(f"Mosaiced S2 Collection {s2_mosaic.getInfo()}")

s2_map.addLayer(s2_mosaic_clipped, vis_params_s2_rgb, "S2 Mosaiced - Spatial")
s2_map

Mosaiced S2 Collection {'type': 'Image', 'bands': [{'id': 'B1', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 6.5535000000000005}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B2', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 6.5535000000000005}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B3', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 6.5535000000000005}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B4', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 6.5535000000000005}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B5', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 6.5535000000000005}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B6', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 6.5535000000000005}, 'crs'

Map(center=[11.999986, 8.551571], controls=(WidgetControl(options=['position', 'transparent_bg'], position='to…

In [12]:
# Compute median composite the entire S2 collection & clip to Kano extent
s2_median = s2_img_col.median().clip(aoi)
print(f"Mosaiced S2 Collection {s2_mosaic.getInfo()}")

# Compute spectral indices
ndvi = s2_median.normalizedDifference(["B8", "B4"]).rename("ndvi")
ndwi = s2_median.normalizedDifference(["B3", "B8"]).rename("ndwi")

s2_map.addLayer(s2_median, vis_params_s2_fc, "S2 Median Comp")
s2_map.addLayer(ndvi, vis_params_ndvi, "S2 NDVI")
s2_map.addLayer(ndwi, vis_params_ndwi, "S2 NDWI")
s2_map

Mosaiced S2 Collection {'type': 'Image', 'bands': [{'id': 'B1', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 6.5535000000000005}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B2', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 6.5535000000000005}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B3', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 6.5535000000000005}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B4', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 6.5535000000000005}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B5', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 6.5535000000000005}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B6', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 6.5535000000000005}, 'crs'

Map(center=[11.999986, 8.551571], controls=(WidgetControl(options=['position', 'transparent_bg'], position='to…

In [13]:
CHECK

NameError: name 'CHECK' is not defined

## Thematic / Groundwater-Influencing Layers

**Precipitation**

In [14]:
# Average 10 years precipitation
# Data Source: https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY
chirps = (
    ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
    .filterDate('2015-01-01', '2024-12-31')
    .filterBounds(aoi)
    .select('precipitation')
)

# Sum of daily rainfall over 10 years, divided by 10 = mean annual rainfall (mm/year)
rainfall = (chirps.sum()
            .divide(10)
            .rename('rainfall')
            .clip(aoi))

thematic_map_1.addLayer(rainfall.clip(aoi), vis_params_precp, "Rainfall")
thematic_map_1

Map(center=[11.999986, 8.551571], controls=(WidgetControl(options=['position', 'transparent_bg'], position='to…

**Digital ELevation Model (DEM) / Slope**

In [15]:
# Download elevation and compute slope
# Data source: https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003
dem = ee.Image('USGS/SRTMGL1_003')
print(f"Bands in the DEM {dem.bandNames().getInfo()}")


elevation = dem.select('elevation')
slope = ee.Terrain.slope(elevation) # Compute slope in degrees


thematic_map_1.addLayer(dem.clip(aoi), vis_params_dem, "DEM")
thematic_map_1.addLayer(slope.clip(aoi), vis_params_slope, "Slope")
thematic_map_1

Bands in the DEM ['elevation']


Map(center=[11.999986, 8.551571], controls=(WidgetControl(options=['position', 'transparent_bg'], position='to…

**Land Use Land Cover**

In [16]:
# ESA world cover
# Data Source: https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200
esa_worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().clip(aoi)
land_cover = esa_worldcover.rename('lulc')

thematic_map_1.addLayer(esa_worldcover, vis_params_esa_lc, "Land Cover")
thematic_map_1

Map(bottom=30867.0, center=[11.999986, 8.551571], controls=(WidgetControl(options=['position', 'transparent_bg…

**Soil Texture**

In [17]:
# Data source: https://developers.google.com/earth-engine/datasets/catalog/OpenLandMap_SOL_SOL_TEXTURE-CLASS_USDA-TT_M_v02
# Band 'b0' corresponds to the surface (0 cm) soil texture class
soil_texture = (
    ee.Image('OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02')
    .select('b0')
    .rename('soil_texture')
    .clip(aoi)
)

thematic_map_1.addLayer(soil_texture, vis_params_soil, "Soil Texture")
thematic_map_1

Map(bottom=30867.0, center=[11.999986, 8.551571], controls=(WidgetControl(options=['position', 'transparent_bg…

**Drainage Density**

In [18]:
# Drainage Density
# Data Source: https://developers.google.com/earth-engine/datasets/catalog/MERIT_Hydro_v1_0_1
merit_hydro = ee.Image('MERIT/Hydro/v1_0_1').clip(aoi)

# 'upa' = upstream drainage area (km^2). flow-accumulation proxy
flow_accumulation = merit_hydro.select('upa').rename('flow_accumulation')

# Threshold to extract a binary stream network
STREAM_THRESHOLD_KM2 = 5
stream_binary = flow_accumulation.gt(STREAM_THRESHOLD_KM2).rename('stream_binary')

# Drainage density proxy
drainage_kernel = ee.Kernel.circle(radius=1500, units='meters', normalize=False)
drainage_density = (
    stream_binary.reduceNeighborhood(reducer=ee.Reducer.sum(), kernel=drainage_kernel)
    .rename('drainage_density')
    .clip(aoi)
)

# Check min & max values
drainage_density_stats = drainage_density.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=aoi,
    scale=90,
    maxPixels=1e13
)

print(drainage_density_stats.getInfo())

# Visualize
thematic_map_1.addLayer(drainage_density, vis_params_drainage, "Drainage Density")
thematic_map_1

{'drainage_density_max': 116, 'drainage_density_min': 0}


Map(bottom=30867.0, center=[11.999986, 8.551571], controls=(WidgetControl(options=['position', 'transparent_bg…

**Topographic Wetness Index (TWI)**

In [19]:
# Slope from degrees to radian
slope_rad = slope.multiply(np.pi / 180)

# Catchment area proxy  from MERIT Hydro 
# (m^2) to (km^2)
catchment_area_m2 = flow_accumulation.multiply(1e6) 

# TWI = ln( catchment area / tan(slope) ), tan(slope) 
twi = (
    catchment_area_m2.divide(slope_rad.tan().max(0.001))
    .log()
    .rename('twi')
    .clip(aoi)
)


# Check min & max values
twi_stats = twi.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=aoi,
    scale=90,
    maxPixels=1e13
)

print(twi_stats.getInfo())


thematic_map_1.addLayer(twi, vis_params_twi, "Topographic Wetness Index (TWI)")
thematic_map_1

{'twi_max': 30.543950863012704, 'twi_min': 9.091400624035863}


Map(bottom=30867.0, center=[11.999986, 8.551571], controls=(WidgetControl(options=['position', 'transparent_bg…

**Distance to Stream/Water**

In [20]:
# Euclidean Distance to stream/ water bodies
distance_to_streams = (
    stream_binary.selfMask()
    .fastDistanceTransform()
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt())
    .rename('dist_to_streams')
    .clip(aoi)
)


In [22]:
# Summary of thematic layers

"""
rainfall
elevation
slope
land_cover
soil_texture
drainage_density
twi
"""

'\nrainfall\nelevation\nslope\nland_cover\nsoil_texture\ndrainage_density\ntwi\n'

In [21]:
STOP

NameError: name 'STOP' is not defined

## Resample Layers

In [23]:
def resample_to_30m(image, method='bilinear'):
    '''
    Resamples an Earth Engine image to a 30m target scale.
    method: 'bilinear' (default) for continuous layers, 
            'nearest' for categorical layers.
    '''
    if method == 'nearest':
        # Nearest neighbor is default
        return image.reproject(crs=image.projection(), scale=30)
    else:
        # Continuous variables
        return image.resample('bilinear').reproject(crs=image.projection(), scale=30)

In [24]:
# Continuous Layers (Bilinear) 
rainfall_30m         = resample_to_30m(rainfall, method='bilinear')
elevation_30m        = resample_to_30m(elevation, method='bilinear')
slope_30m            = resample_to_30m(slope, method='bilinear')
drainage_density_30m = resample_to_30m(drainage_density, method='bilinear')
twi_30m              = resample_to_30m(twi, method='bilinear')

# Categorical Layers (Nearest Neighbor
land_cover_30m   = resample_to_30m(land_cover, method='nearest')
soil_texture_30m = resample_to_30m(soil_texture, method='nearest')

## Reclassify Thematic Layers

**THIS CAN BE DONE IN QGIS USING RASTER CALCULATOR**

In [ ]:
def get_percentile_breaks(image, band, geometry, scale):
    '''Return the 20th, 40th, 60th and 80th percentile values of a band over a geometry.'''
    stats = image.select(band).reduceRegion(
        reducer=ee.Reducer.percentile([20, 40, 60, 80]),
        geometry=geometry,
        scale=scale,
        maxPixels=1e13,
        bestEffort=True,
        tileScale=4,
    ).getInfo()
    return [
        stats[f'{band}_p20'],
        stats[f'{band}_p40'],
        stats[f'{band}_p60'],
        stats[f'{band}_p80'],
    ]


def reclass_continuous(image, band, breaks, inverse=False):
    '''
    Reclassify a continuous band into scores 1-5 using percentile breaks.
    inverse=True is used when LOWER raw values represent HIGHER groundwater suitability
    (e.g. elevation, slope, drainage density, distance to streams).
    '''
    b20, b40, b60, b80 = breaks
    score = (
        ee.Image(1)
        .where(image.gte(b20), 2)
        .where(image.gte(b40), 3)
        .where(image.gte(b60), 4)
        .where(image.gte(b80), 5)
    )
    if inverse:
        score = ee.Image(6).subtract(score)
    return score.rename(band + '_score').updateMask(image.mask())


In [ ]:
# Continuous layers reclassify



# Export Layers/ Data

In [ ]:
#export_image_to_drive(image: ee.Image, description: str, folder: str, extent: ee.Geometry, scale: int, crs: str = 'EPSG:4326')
elevation_export = export_image_to_drive(elevation_30m.clip(aoi), "Elevation_Kano_30m", "3MTT_C3_Grp2", aoi, 30, "EPSG:32632")
slope_export = export_image_to_drive(slope_30m.clip(aoi), "Slope_Kano_30m", "3MTT_C3_Grp2", aoi, 30, "EPSG:32632")
soil_texture_export = export_image_to_drive(soil_texture_30m.clip(aoi), "Soil_Texture_Kano_30m", "3MTT_C3_Grp2", aoi, 30, "EPSG:32632")
land_cover_export = export_image_to_drive(land_cover_30m.clip(aoi), "Land_Cover_Kano_30m", "3MTT_C3_Grp2", aoi, 30, "EPSG:32632")

#s2_median_export = export_image_to_drive(s2_median.select(["B2", "B3", "B4", "B8"]), "Kano_S2_Median_", "3MTT_C3_Grp2", aoi, 10, "EPSG:32632")

In [ ]:
print(elevation_export.status())
print(slope_export.status())
#s2_median_export.status()

{'state': 'RUNNING',
 'description': 'Kano_S2_Median_',
 'priority': 100,
 'creation_timestamp_ms': 1784926877061,
 'update_timestamp_ms': 1784926884458,
 'start_timestamp_ms': 1784926881019,
 'task_type': 'EXPORT_IMAGE',
 'attempt': 1,
 'id': 'EBM5I4MBTGEDZENV5JOHC3L5',
 'name': 'projects/487673000658/operations/EBM5I4MBTGEDZENV5JOHC3L5'}